# HH->bbtautau: Data Exploration with Plotly
Quick-look setup for exploring tensor-based features interactively.

In [1]:
# add root of module into sys.path to be able to import bbTT related code
import sys
from pathlib import Path

# notebook is in src/bbTT/plotly -> project root is 3 levels up
project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root / "src"))

In [2]:
from __future__ import annotations

import numpy as np
import torch
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
import pandas as pd
import math
import ipywidgets as widgets
from IPython.display import display
# Notebook rendering: makes plotly figures show inline
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import bbTT.plotly.preparation as prep
import os

from bbTT.configs.full_config import FullConfig
from bbTT.data_handling import io
full_config = FullConfig()


[15:25:32] INFO L:47-find_datasets: Start searching for datasets:


# Load Input Feautres and DNN scores


In [3]:
# load Dnn score
EVAL_DIR = Path("/data/dust/user/wiedersb/HH_DNN/evaluation/")
name = "6ac9c66ff0_training_file.pt"
fold = 0

_parts = name.split("_")
cache_name = _parts[0]
model_name = _parts[1]

path = EVAL_DIR / name
scores = torch.load(path, weights_only=True)[fold]

In [4]:
# load input features corresponding to your scores
# TODO make this not depending on config OR make this depending on config of model
events = io.get_data(full_config.dataset_config, ignore_cache=False, save_cache=True)
labels_continous = full_config.dataset_config.continuous_features
labels_categorical = full_config.dataset_config.categorical_features

[15:25:33] INFO L:319-create_era_caches: Cache already present for era 22pre
[15:25:33] INFO L:390-load_and_merge_eras: Loading cached data for era 22pre
[15:25:33] DEBUG L:62-load_era: Loading cache from: /data/dust/user/wiedersb/HH_DNN/cache/6ac9c66ff0/22pre.pkl


In [5]:
# convert to pandas
df_scores = prep.build_scores_frame(scores, (list(full_config.dataset_config.target_map.keys())))
df_events = prep.build_feature_frame(events, scores, labels_continous, labels_categorical)

# merge both together over event_id
# how='inner'   # only rows where the key exists in BOTH frames
# how='left'    # keep all of df1, NaN where df2 has no match
# how='right'   # keep all of df2, NaN where df1 has no match
# how='outer'   # keep everything from both, NaN where no match
df = pd.merge(df_events, df_scores, on="event_id", how="outer", suffixes=("_input", "_scores"))

# check if any mismatch happened
#if df["label_scores"].isna().any(axis=0):
#    raise ValueError("There are events that exist in input but not in scores")
del df_scores
del df_events
del scores
del events


In [6]:
# construct physics relevant quantities and add them to df
import bbTT.plotly.physics as physics
columns_to_compute = (
    "bjet1", "bjet2", "fatjet", "htt", "hbb", "httfatjet", "htthbb", "vis_tau1", "vis_tau2"
)

physics.compute_derived(df=df, column_names = columns_to_compute)
    


/afs/desy.de/user/w/wiedersb/xxl/pytorch_network_playground/src/bbTT/plotly/physics.py:23: UserWarning: Warning: 882844 NaN values in fatjet_eta, replacing with 0.
  warnings.warn(f"Warning: {num_nan} NaN values in {column_name}, replacing with 0.")
/data/dust/user/wiedersb/pyenv_virtualenvs/ml_torch/lib64/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/afs/desy.de/user/w/wiedersb/xxl/pytorch_network_playground/src/bbTT/plotly/physics.py:23: UserWarning: Warning: 882844 NaN values in httfatjet_eta, replacing with 0.
  warnings.warn(f"Warning: {num_nan} NaN values in {column_name}, replacing with 0.")
/data/dust/user/wiedersb/pyenv_virtualenvs/ml_torch/lib64/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/data/dust/user/wiedersb/pyenv_virtualenvs/ml_torch/lib64/python3.9

# Overview of Columns:

In [7]:
import bbTT.plotly.column_overview_widget as overview
display(overview.show_columns_overview(df))

# Plotting 1D


In [31]:
import bbTT.plotly.hist_1D as h1d 
import importlib 
importlib.reload(h1d)


# processes are stacked together
process_definitions = {
    "Signal*": {21101},
    "tt:fh": {1100},
    "tt:sl": {1200},
    "tt:dl": {1300},
    "DY": {
       51661, 51663, 51664, 51665, 51666,
       51667, 51668, 51670, 51671, 51672, 51673, 51674, 51675, 51677,
       51679, 51680, 51681, 51682, 51683, 51684, 51686, 51687, 51688,
       51689, 51690, 51691, 51694, 51695, 51700, 51701, 51703, 51704,
       51706, 51707, 51709, 51710, 51712, 51715, 51721, 51722, 51724,
       51725, 51727, 51728, 51730, 51731, 51733, 51734, 51736, 51737}
    # add DY, QCD, etc. here
}
process_colors = {
    "Signal*": "black",
    "tt:fh": "#ffa500", # orange
    "tt:sl": "#ff0000", # red
    "tt:dl": "#0000ff", # blue
    "DY": "#008000" # green
}

family_hatch = {
    "tt":"/",
    "DY": "."
}

# cuts one can toggle, use dict
predifined_cuts = {"high_dy":"dnn_score_dy > 0.5"}

column_overrides = {
    "dnn_score_dy": {"range": (0,1)},
    "dnn_score_tt": {"range": (0,1)},
    "dnn_score_hh": {"range": (0,1)},
    
    
}

plot_pairs = [
    {"x": "dnn_score_tt", "y": "dnn_score_dy", "cut": ""}]

columns = [
        f"{col}_{att}"
        for att in ("pt", "e", "mass", "eta", "phi")
        for col in ["bjet1", "bjet2", "fatjet", "htt", "hbb", "httfatjet", "htthbb", "vis_tau1", "vis_tau2"]
    ]

columns = [f for f in df.keys() if f.startswith("vis_tau1")]


plotter_1d = h1d.HistogramPlotter1D(
    df, 
    predifined_cuts, 
    process_column="pid",
    process_definitions=process_definitions,
    columns=columns,
    n_bins_range=(5, 30),
    n_bins=10,
    n_cols=4,
    clip_quantiles=(0.01, 0.99),
    column_overrides=None,
    include_other=True,
    weight_column_name = "event_weights",
    process_colors=process_colors,
    opacity_filling=0.1,
    family_hatch=family_hatch,
)


plotter_1d.show()


FigureWidget({
    'data': [{'fill': 'tozeroy',
              'fillcolor': 'rgba(0,128,0,0.1)',
              'fillpattern': {'fgcolor': '#008000', 'shape': '.', 'size': 6, 'solidity': 0.3},
              'legendgroup': 'DY',
              'line': {'color': '#008000', 'shape': 'hv', 'width': 1.5},
              'mode': 'lines',
              'name': 'DY',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8a988553-5f1f-4581-9680-eec56031836d',
              'visible': True,
              'x': {'bdata': ('vEkMXtI1OMAAoBpvnMf0v7z1KNDenD' ... 'U4cwVhQIUWWUue4mNAfGo8Xsm/ZkA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('AAAAgL45bUAAAABAOF57QAAAAACISZ' ... 'AAYMxMQAAAAABAtD1AAAAAACCWMkA='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'fill': 'tozeroy',
              'fillcolor': 'rgba(255,165,0,0.1)',
              'fillpattern': {'fgcolor': '#ffa500', 'shape': '/', '

# Plotting 2D

In [22]:
import bbTT.plotly.hist_2D as h2d

# togglable cuts
# currently buggy


predifined_cuts = {
    "hh_pids" : "pid == 21101", 
    "dy_pids" : "&".join([f"pid == {pid}" for pid in list(map(str,process_definitions["DY"]))]) , 
    "tt_pids" : "&".join([f"pid == {pid}" for pid in ("1100", "1200", "1300")]), 
    }

# column specific settings
column_overrides = {"dnn_score_hh": {"range": (0,1)}}

# default plots with individual cuts
plot_pairs = [
    {"x": "dnn_score_hh", "y": "htthbb_mass", "cut": ""},
    {"x": "dnn_score_hh", "y": "htt_mass", "cut": ""},
    {"x": "dnn_score_hh", "y": "hbb_mass", "cut": ""},


]


plotter_2d = h2d.HistogramPlotter2D(
        predefined_cuts,
        column_overrides=None,
        n_bins_default=30,
        n_bins_range=(5, 100),
        clip_quantiles=(0.01, 0.99),
        n_cols=2,
        plot_pairs: list[dict] = None,

    
    df, predifined_cuts, column_overrides=column_overrides, plot_pairs = plot_pairs)
plotter_2d.show()